# SMILES-2026 Hallucination Detection

Detect whether Qwen2.5-0.5B's response is **hallucinated (1)** or **truthful (0)** from the model's own hidden states.

**Pipeline:** load Qwen → extract per-layer hidden states for each `prompt+response` → aggregate into one feature vector per sample → train a small MLP probe → evaluate → generate `predictions.csv`.

Runtime tips for Colab T4:
- **Runtime → Change runtime type → GPU (T4)** before running.
- Feature extraction takes ~20–30 min once; results are cached to `.npy` files so a Colab restart doesn't re-run it.
- This notebook is **self-contained** — cell 2 writes the three student files (aggregation, probe, splitting) into the cloned repo, overwriting the upstream placeholders.

## 1. Setup (Colab)

Clones the repo and installs deps. Skips silently if you're running locally and these are already present.

In [6]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO = 'SMILES-2026-Hallucination-Detection'
REPO_URL = 'https://github.com/ahdr3w/SMILES-HALLUCINATION-DETECTION.git'

if IN_COLAB:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

cwd: /content/SMILES-2026-Hallucination-Detection/SMILES-2026-Hallucination-Detection
files: ['.git', '.gitignore', 'LICENSE', 'README.md', 'aggregation.py', 'data', 'evaluate.py', 'model.py', 'probe.py', 'requirements.txt', 'solution.py', 'splitting.py']


## 2. Write the three student files

Overwrites the upstream placeholders with the competition-strength implementations.

In [7]:
%%writefile aggregation.py
"""aggregation.py - last-token concat from 3 mid/late layers + geometric features."""
from __future__ import annotations
import math
import torch
import torch.nn.functional as F

SELECTED_LAYERS_FRAC = (0.5, 0.75, 1.0)

def _selected_layer_indices(n_layers: int) -> list[int]:
    return [max(1, int(round(f * (n_layers - 1)))) for f in SELECTED_LAYERS_FRAC]

def _last_real_index(attention_mask: torch.Tensor) -> int:
    real = attention_mask.nonzero(as_tuple=False)
    return int(real[-1].item())

def aggregate(hidden_states, attention_mask):
    n_layers = hidden_states.size(0)
    last_pos = _last_real_index(attention_mask)
    layer_idx = _selected_layer_indices(n_layers)
    parts = [hidden_states[i, last_pos] for i in layer_idx]
    return torch.cat(parts, dim=0)

def extract_geometric_features(hidden_states, attention_mask):
    n_layers, _, _ = hidden_states.shape
    last_pos = _last_real_index(attention_mask)
    last_token = hidden_states[:, last_pos, :]
    norms = last_token.norm(dim=-1)
    cos = F.cosine_similarity(last_token[:-1], last_token[1:], dim=-1)
    seq_len = float(attention_mask.sum().item())
    log_len = torch.tensor(
        [math.log1p(seq_len)],
        dtype=hidden_states.dtype,
        device=hidden_states.device,
    )
    return torch.cat([norms, cos, log_len], dim=0)

def aggregation_and_feature_extraction(hidden_states, attention_mask, use_geometric=False):
    agg = aggregate(hidden_states, attention_mask)
    if use_geometric:
        geo = extract_geometric_features(hidden_states, attention_mask)
        return torch.cat([agg, geo], dim=0)
    return agg


Overwriting aggregation.py


In [8]:
%%writefile probe.py
"""probe.py - regularized MLP with dropout, weight decay, early stopping."""
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def _device():
    if torch.cuda.is_available(): return torch.device('cuda')
    if torch.backends.mps.is_available(): return torch.device('mps')
    return torch.device('cpu')

class HallucinationProbe(nn.Module):
    HIDDEN_1 = 512
    HIDDEN_2 = 128
    DROPOUT = 0.3
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    MAX_EPOCHS = 300
    PATIENCE = 20
    INTERNAL_VAL_FRAC = 0.1
    SEED = 42

    def __init__(self):
        super().__init__()
        self._net = None
        self._scaler = StandardScaler()
        self._threshold = 0.5
        self._device = _device()

    def _build_network(self, input_dim):
        self._net = nn.Sequential(
            nn.Linear(input_dim, self.HIDDEN_1), nn.ReLU(), nn.Dropout(self.DROPOUT),
            nn.Linear(self.HIDDEN_1, self.HIDDEN_2), nn.ReLU(), nn.Dropout(self.DROPOUT),
            nn.Linear(self.HIDDEN_2, 1),
        )
        self._net.to(self._device)

    def forward(self, x):
        if self._net is None:
            raise RuntimeError('Network not built. Call fit() first.')
        return self._net(x).squeeze(-1)

    def fit(self, X, y):
        X_scaled = self._scaler.fit_transform(X)
        self._build_network(X_scaled.shape[1])
        if len(y) > 50 and len(np.unique(y)) > 1:
            X_tr, X_va, y_tr, y_va = train_test_split(
                X_scaled, y, test_size=self.INTERNAL_VAL_FRAC,
                random_state=self.SEED, stratify=y,
            )
        else:
            X_tr, X_va, y_tr, y_va = X_scaled, X_scaled, y, y
        X_tr_t = torch.from_numpy(X_tr).float().to(self._device)
        y_tr_t = torch.from_numpy(y_tr.astype(np.float32)).to(self._device)
        X_va_t = torch.from_numpy(X_va).float().to(self._device)
        y_va_t = torch.from_numpy(y_va.astype(np.float32)).to(self._device)
        n_pos = int(y_tr.sum())
        n_neg = len(y_tr) - n_pos
        pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=self._device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.Adam(self.parameters(), lr=self.LR, weight_decay=self.WEIGHT_DECAY)
        best_val = float('inf'); best_state = None; epochs_no_improve = 0
        for _ in range(self.MAX_EPOCHS):
            self.train(); optimizer.zero_grad()
            loss = criterion(self(X_tr_t), y_tr_t); loss.backward(); optimizer.step()
            self.eval()
            with torch.no_grad():
                val_loss = criterion(self(X_va_t), y_va_t).item()
            if val_loss < best_val - 1e-4:
                best_val = val_loss
                best_state = {k: v.detach().clone() for k, v in self.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= self.PATIENCE: break
        if best_state is not None: self.load_state_dict(best_state)
        self.eval()
        return self

    def fit_hyperparameters(self, X_val, y_val):
        probs = self.predict_proba(X_val)[:, 1]
        candidates = np.unique(np.concatenate([probs, np.linspace(0.0, 1.0, 101)]))
        best_threshold = 0.5; best_f1 = -1.0
        for t in candidates:
            score = f1_score(y_val, (probs >= t).astype(int), zero_division=0)
            if score > best_f1:
                best_f1 = score; best_threshold = float(t)
        self._threshold = best_threshold
        return self

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self._threshold).astype(int)

    def predict_proba(self, X):
        X_scaled = self._scaler.transform(X)
        X_t = torch.from_numpy(X_scaled).float().to(self._device)
        self.eval()
        with torch.no_grad():
            prob_pos = torch.sigmoid(self(X_t)).cpu().numpy()
        return np.stack([1.0 - prob_pos, prob_pos], axis=1)


Overwriting probe.py


In [9]:
%%writefile splitting.py
"""splitting.py - stratified 5-fold with inner train/val carve."""
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

N_SPLITS = 5
VAL_FRAC = 0.2
SEED = 42

def split_data(y, df=None, test_size=0.15, val_size=0.15, random_state=SEED):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=random_state)
    idx = np.arange(len(y))
    folds = []
    for trainval_idx, test_idx in skf.split(idx, y):
        idx_train, idx_val = train_test_split(
            trainval_idx, test_size=VAL_FRAC,
            random_state=random_state, stratify=y[trainval_idx],
        )
        folds.append((idx_train, idx_val, test_idx))
    return folds


Overwriting splitting.py


## 3. Imports & config

In [10]:
import importlib, time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

import aggregation, probe, splitting
importlib.reload(aggregation); importlib.reload(probe); importlib.reload(splitting)
from aggregation import aggregation_and_feature_extraction
from evaluate import print_summary, run_evaluation, save_predictions, save_results
from model import MAX_LENGTH, get_model_and_tokenizer
from probe import HallucinationProbe
from splitting import split_data

DATA_FILE        = './data/dataset.csv'
TEST_FILE        = './data/test.csv'
OUTPUT_FILE      = 'results.json'
PREDICTIONS_FILE = 'predictions.csv'
BATCH_SIZE       = 4
USE_GEOMETRIC    = True

CACHE_DIR  = '/content' if 'google.colab' in sys.modules else '.'
X_CACHE    = f'{CACHE_DIR}/X_train.npy'
Y_CACHE    = f'{CACHE_DIR}/y_train.npy'
XT_CACHE   = f'{CACHE_DIR}/X_test.npy'

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'device         : {device}')
print(f'max_length     : {MAX_LENGTH}')
print(f'use_geometric  : {USE_GEOMETRIC}')
print(f'cache_dir      : {CACHE_DIR}')

device         : cuda
max_length     : 512
use_geometric  : True
cache_dir      : /content


## 4. Load training data

In [11]:
df = pd.read_csv(DATA_FILE)
all_texts  = [f"{row['prompt']}{row['response']}" for _, row in df.iterrows()]
all_labels = np.array([int(float(h)) for h in df['label']])

print(f'rows           : {len(df)}')
print(f'class balance  : {dict(df["label"].value_counts().sort_index())}')
row0 = df.iloc[0]
print('\n-- prompt (first 300 chars) --')
print(row0['prompt'][:300])
print('\n-- response (first 200 chars) --')
print(row0['response'][:200])
print(f'\nlabel: {int(row0["label"])}')

rows           : 689
class balance  : {0.0: np.int64(206), 1.0: np.int64(483)}

-- prompt (first 300 chars) --
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or enginee

-- response (first 200 chars) --
An architect or engineer has a direct relationship with the subcontractor.<|endoftext|>

label: 1


## 5. Load Qwen2.5-0.5B

In [12]:
model, tokenizer = get_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.to(device)
print('model loaded.')

[Model] Loading 'Qwen/Qwen2.5-0.5B' ...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model loaded.


## 6. Extract training features

One forward pass per sample, batched. Cached to disk so a restart doesn't re-run it.

In [13]:
def extract_features(texts):
    feats = []
    for start in tqdm(range(0, len(texts), BATCH_SIZE), unit='batch'):
        batch = texts[start:start + BATCH_SIZE]
        enc = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH,
        )
        input_ids      = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        hidden = torch.stack(outputs.hidden_states, dim=1).float()  # (B, L, S, H)

        for i in range(hidden.size(0)):
            f = aggregation_and_feature_extraction(
                hidden[i], attention_mask[i], use_geometric=USE_GEOMETRIC,
            )
            feats.append(f.detach().cpu().numpy())
    return np.vstack(feats)

if os.path.exists(X_CACHE) and os.path.exists(Y_CACHE):
    X = np.load(X_CACHE)
    y = np.load(Y_CACHE)
    extract_time = 0.0
    print(f'loaded cached features: {X.shape}')
else:
    t0 = time.time()
    X = extract_features(all_texts)
    y = all_labels
    extract_time = time.time() - t0
    np.save(X_CACHE, X)
    np.save(Y_CACHE, y)
    print(f'extracted in {extract_time:.1f}s -> {X.shape}, cached to {X_CACHE}')

  0%|          | 0/173 [00:00<?, ?batch/s]

extracted in 164.1s -> (689, 2738), cached to /content/X_train.npy


## 7. Splits

In [14]:
splits = split_data(y, df)
print(f'folds: {len(splits)}')
for i, (tr, va, te) in enumerate(splits):
    va_n = len(va) if va is not None else 0
    print(f'  fold {i+1}: train={len(tr)}  val={va_n}  test={len(te)}')

folds: 5
  fold 1: train=440  val=111  test=138
  fold 2: train=440  val=111  test=138
  fold 3: train=440  val=111  test=138
  fold 4: train=440  val=111  test=138
  fold 5: train=441  val=111  test=137


## 8. Train probe + evaluate (per fold)

In [15]:
fold_results = run_evaluation(splits, X, y, HallucinationProbe)
print_summary(fold_results, X.shape[1], len(X), extract_time)
save_results(fold_results, X.shape[1], len(X), extract_time, OUTPUT_FILE)


──────────────────────────────────────────────────
  Fold 1/5  —  train=440  val=111  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 72.27%  F1: 83.20%  AUROC: 75.11%
  Probe val  — Acc: 76.58%  F1: 85.71%  AUROC: 73.89%
  Probe test — Acc: 68.84%  F1: 81.22%  AUROC: 67.36%

──────────────────────────────────────────────────
  Fold 2/5  —  train=440  val=111  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 70.91%  F1: 81.29%  AUROC: 75.70%
  Probe val  — Acc: 73.87%  F1: 83.43%  AUROC: 70.20%
  Probe test — Acc: 73.91%  F1: 83.49%  AUROC: 72.87%

──────────────────────────────────────────────────
  Fold 3/5  —  train=440  val=111  test=138
──────────────────────────────────────────────────
  Baseline  — Acc: 70.29%  F1: 82.55%
  Probe train — Acc: 73.86%  F1: 84.18%  AUROC: 78.23%
  Probe val  — Acc: 72.97%  F1: 83.70%  AUROC: 67.13%
  Probe te

## 9. Extract test features

In [16]:
df_test    = pd.read_csv(TEST_FILE)
test_texts = [f"{row['prompt']}{row['response']}" for _, row in df_test.iterrows()]
test_ids   = df_test.index

if os.path.exists(XT_CACHE):
    X_test = np.load(XT_CACHE)
    print(f'loaded cached test features: {X_test.shape}')
else:
    X_test = extract_features(test_texts)
    np.save(XT_CACHE, X_test)
    print(f'extracted test features: {X_test.shape}, cached to {XT_CACHE}')

  0%|          | 0/25 [00:00<?, ?batch/s]

extracted test features: (100, 2738), cached to /content/X_test.npy


## 10. Refit probe on all train+val, generate predictions.csv

In [17]:
idx_non_test = np.unique(np.concatenate([
    np.concatenate([tr, va]) if va is not None else tr
    for tr, va, _ in splits
]))

final_probe = HallucinationProbe()
final_probe.fit(X[idx_non_test], y[idx_non_test])
save_predictions(final_probe, X_test, test_ids, PREDICTIONS_FILE)

Predictions saved to 'predictions.csv'  (100 samples)


## 11. Download artifacts (Colab only)

In [18]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(PREDICTIONS_FILE)
    files.download(OUTPUT_FILE)
else:
    print(f'wrote {PREDICTIONS_FILE} and {OUTPUT_FILE} to {os.getcwd()}')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>